In [71]:
import pandas as pd
import numpy as np
 
df = pd.read_csv("sales_messy.csv")
print(df.shape)        # (25, 5)
print(df.head(10))     # take a real look first


(25, 5)
          date   stall    fruit qty price
0   2024-06-03   North    apple  12    30
1   2024-06-03   north    Apple   5    30
2   03/06/2024   South  banana   20   ₹10
3   2024-06-04   NORTH    APPLE   9    30
4   2024-06-04   South   orange   7   ₹20
5   2024-06-05   North   Banana  18    10
6  June 5 2024   south    grape   8    50
7   2024-06-05   South    mango   3    40
8   2024-06-06  North     apple  14    30
9   2024-06-06   South  orange    5    20


In [72]:
print(df.info())              # column types + how many non-blank
print(df.describe(include="all"))   # quick stats for every column
print(df.isna().sum())        # blanks per column
print(df.duplicated().sum())  # how many fully-repeated rows


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    24 non-null     object
 1   stall   25 non-null     object
 2   fruit   25 non-null     object
 3   qty     24 non-null     object
 4   price   24 non-null     object
dtypes: object(5)
memory usage: 1.1+ KB
None
              date  stall  fruit qty price
count           24     25     25  24    24
unique          12      6     12  19     8
top     2024-06-03  South  apple  12    30
freq             3     11      5   2     8
date     1
stall    0
fruit    0
qty      1
price    1
dtype: int64
1


In [73]:
#a
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    24 non-null     object
 1   stall   25 non-null     object
 2   fruit   25 non-null     object
 3   qty     24 non-null     object
 4   price   24 non-null     object
dtypes: object(5)
memory usage: 1.1+ KB
None


In [33]:
#b
print(df.isna().sum())

date     1
stall    0
fruit    0
qty      1
price    1
dtype: int64


In [34]:
#c
print(df.duplicated().sum())

1


In [35]:
#d
print(df["fruit"].nunique())

12


In [36]:
# price: remove the ₹ symbol, then convert to a number
df["price"] = pd.to_numeric(df["price"].str.replace("₹", "", regex=False),
                            errors="coerce")
 
# qty: convert; "ten" and blanks become NaN
df["qty"] = pd.to_numeric(df["qty"], errors="coerce")
 
# date: parse many formats; unreadable ones become NaT
df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")
print(df.dtypes)


date     datetime64[ns]
stall            object
fruit            object
qty             float64
price           float64
dtype: object


In [37]:
#a
df["price"] = pd.to_numeric(
    df["price"].astype(str).str.replace("₹", "", regex=False),
    errors="coerce"
)

print(df["price"].isna().sum())

1


In [38]:
#b
df["qty"] = pd.to_numeric(df["qty"], errors="coerce")

print(df["qty"].isna().sum())

2


In [39]:
#c
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce",
    format="mixed"
)

print(df["date"].isna().sum())

1


In [40]:
print(df.duplicated().sum())          # 1 duplicate row
print(df[df.duplicated(keep=False)])  # show all copies
 
df = df.drop_duplicates()
print(df.shape)                       # one row fewer


1
         date  stall  fruit   qty  price
0  2024-06-03  North  apple  12.0   30.0
13 2024-06-03  North  apple  12.0   30.0
(24, 5)


In [41]:
#a
print(df[df.duplicated(keep=False)])

df = df.drop_duplicates()

print(df.shape)

Empty DataFrame
Columns: [date, stall, fruit, qty, price]
Index: []
(24, 5)


In [42]:

Dropping duplicates prevents the same sale from being counted multiple times, which would inflate the total revenue.

In [43]:
print(df["stall"].unique())   # 6 spellings of 2 real stalls!
 
df["stall"] = df["stall"].str.strip().str.title()
df["fruit"] = df["fruit"].str.strip().str.lower()
 
print(df["stall"].unique())   # North, South
print(df["fruit"].value_counts())


['North' 'north' 'South' 'NORTH' 'south' 'North ']
['North' 'South']
fruit
apple     6
banana    5
orange    4
grape     4
mango     4
appel     1
Name: count, dtype: int64


In [44]:
print(df["stall"].unique())

df["stall"] = df["stall"].str.strip().str.title()

print(df["stall"].unique())

['North' 'South']
['North' 'South']


In [45]:
#b
df["fruit"] = df["fruit"].str.strip().str.lower()

print(df["fruit"].value_counts())

fruit
apple     6
banana    5
orange    4
grape     4
mango     4
appel     1
Name: count, dtype: int64


In [46]:
#c
df["fruit"] = df["fruit"].replace("appel", "apple")

print(df["fruit"].value_counts())

fruit
apple     7
banana    5
orange    4
grape     4
mango     4
Name: count, dtype: int64


In [47]:
print(df.isna().sum())        # the true picture now
 
# example decisions:
df = df.dropna(subset=["date"])          # a sale with no date is unusable
df["price"] = df["price"].fillna(df["price"].median())   # fair estimate


date     1
stall    0
fruit    0
qty      2
price    1
dtype: int64


In [48]:
#a
print(df.isna().sum())

date     0
stall    0
fruit    0
qty      2
price    0
dtype: int64


In [49]:
#c
# date: a sale without a date cannot be placed in time, so remove it
df = df.dropna(subset=["date"])

# price: fill missing prices with the median price as a fair estimate
df["price"] = df["price"].fillna(df["price"].median())

# qty: fill missing quantities with the median quantity
df["qty"] = df["qty"].fillna(df["qty"].median())

print(df.isna().sum())

date     0
stall    0
fruit    0
qty      0
price    0
dtype: int64


In [50]:
Q1 = df["qty"].quantile(0.25)
Q3 = df["qty"].quantile(0.75)
IQR = Q3 - Q1
low, high = Q1 - 1.5*IQR, Q3 + 1.5*IQR
print(low, high)                                  # -4.0  24.0
print(df[(df["qty"] < low) | (df["qty"] > high)]["qty"].tolist())


-4.0 24.0
[25.0, -5.0, 9999.0]


In [51]:
#a
Q1 = df["qty"].quantile(0.25)
Q3 = df["qty"].quantile(0.75)

IQR = Q3 - Q1

low = Q1 - 1.5 * IQR
high = Q3 + 1.5 * IQR

print(low, high)

print(df[(df["qty"] < low) | (df["qty"] > high)]["qty"].tolist())

-4.0 24.0
[25.0, -5.0, 9999.0]


In [52]:
#b
# -5 is impossible for a sales quantity, so mark it as missing
df.loc[df["qty"] < 0, "qty"] = np.nan

# 9999 is clearly a data-entry error, so mark it as missing
df.loc[df["qty"] == 9999, "qty"] = np.nan

# 25 is a large but possible sale, so keep it

print(df["qty"].isna().sum())

2


In [53]:
#c
print(df[df["price"] <= 0]["price"])

Series([], Name: price, dtype: float64)


In [54]:
print((df["qty"] > 0).all())        # should every quantity be positive? 
print((df["price"] > 0).all())      # every price positive?
print(df["fruit"].isin(["apple","banana","orange","mango","grape","lemon"]).all())


False
True
True


In [55]:
#a
print((df["qty"] > 0).all())

print((df["price"] > 0).all())

print(
    df["fruit"].isin(
        ["apple", "banana", "orange", "mango", "grape", "lemon"]
    ).all()
)

False
True
True


In [56]:
#b
# Find rows where qty is missing or not positive
print(df[(df["qty"].isna()) | (df["qty"] <= 0)])

         date  stall   fruit  qty  price
14 2024-06-08  South   apple  NaN   30.0
15 2024-06-08  North  banana  NaN   10.0


In [57]:
# Fill missing quantities with the median quantity
df["qty"] = df["qty"].fillna(df["qty"].median())

# Re-run validation checks
print((df["qty"] > 0).all())
print((df["price"] > 0).all())
print(
    df["fruit"].isin(
        ["apple", "banana", "orange", "mango", "grape", "lemon"]
    ).all()
)

True
True
True


In [74]:
# Fix types
df["price"] = pd.to_numeric(
    df["price"].astype(str).str.replace("₹", "", regex=False),
    errors="coerce"
)

df["qty"] = pd.to_numeric(df["qty"], errors="coerce")

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce",
    format="mixed"
)

# Remove duplicates
df = df.drop_duplicates()

# Standardise text
df["stall"] = df["stall"].str.strip().str.title()

df["fruit"] = (
    df["fruit"]
    .str.strip()
    .str.lower()
    .replace("appel", "apple")
)

# Handle missing values
df = df.dropna(subset=["date"])

df["price"] = df["price"].fillna(df["price"].median())

# Handle outliers
df.loc[df["qty"] < 0, "qty"] = np.nan
df.loc[df["qty"] == 9999, "qty"] = np.nan

df["qty"] = df["qty"].fillna(df["qty"].median())

# Validation checks
print((df["qty"] > 0).all())
print((df["price"] > 0).all())
print(df["fruit"].isin(
    ["apple", "banana", "orange", "mango", "grape", "lemon"]
).all())

# Save cleaned file
df.to_csv("sales_clean.csv", index=False)

True
True
True


## End-of-Day Build – Data Quality Report

| Issue | Where (column) | How many | What I did and why |
|--------|--------|--------|--------|
| Duplicate row | Entire row | 1 row | Removed duplicate to prevent double counting |
| Price stored as text (₹ symbol) | price | 25 rows | Removed ₹ symbol and converted to numeric |
| Quantity stored as text | qty | 25 rows | Converted to numeric using to_numeric() |
| Date stored as text | date | 25 rows | Converted to datetime format |
| Inconsistent stall names | stall | 6 spellings | Standardised with strip() and title() |
| Inconsistent fruit names | fruit | Multiple entries | Standardised with strip() and lower() |
| Typo "appel" | fruit | 1 row | Replaced with "apple" |
| Missing date | date | 1 row | Dropped row because date is required |
| Missing quantity | qty | 2 rows | Filled with median quantity |
| Missing price | price | 1 row | Filled with median price |
| Negative quantity (-5) | qty | 1 row | Treated as an error and replaced |
| Extreme quantity (9999) | qty | 1 row | Treated as a data-entry error and replaced |
| Quantity 25 flagged by IQR | qty | 1 row | Kept because it is a valid large sale |
| Validation failure | qty | 4 rows | Fixed invalid and missing quantities |
### Verdict

The dataset is now suitable for analysis because incorrect data types, duplicate records, inconsistent text, missing values, and invalid outliers have been addressed. All validation checks pass successfully after cleaning. However, some missing values were filled using median values, so those entries contain estimated rather than original recorded data.

sales_clean.csv saved successfully


End-of-day reflection

What clicked today:
I learned how to profile a dataset and systematically clean data-quality issues.

What was tricky:
Deciding whether an outlier was a genuine error or a valid unusual sale.

One question for tomorrow:
How can charts help reveal patterns that are difficult to see in tables?

In [66]:
def find_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)

    IQR = Q3 - Q1

    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR

    return series[(series < low) | (series > high)]

print(find_outliers(df["qty"]))
print(find_outliers(df["price"]))

11    25.0
Name: qty, dtype: float64
Series([], Name: price, dtype: float64)


In [67]:
for fruit, group in df.groupby("fruit"):
    Q1 = group["qty"].quantile(0.25)
    Q3 = group["qty"].quantile(0.75)

    IQR = Q3 - Q1

    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR

    outliers = group[
        (group["qty"] < low) |
        (group["qty"] > high)
    ]

    print(fruit)
    print(outliers[["qty"]])

apple
Empty DataFrame
Columns: [qty]
Index: []
banana
Empty DataFrame
Columns: [qty]
Index: []
grape
Empty DataFrame
Columns: [qty]
Index: []
mango
Empty DataFrame
Columns: [qty]
Index: []
orange
Empty DataFrame
Columns: [qty]
Index: []


In [68]:
fruit_map = {
    "apple": "apple",
    "apple ": "apple",
    "appel": "apple",
    "APPLE": "apple",
    "Apple": "apple"
}

df["fruit"] = (
    df["fruit"]
    .str.strip()
    .replace(fruit_map)
    .str.lower()
)

print(df["fruit"].value_counts())

fruit
apple     7
banana    5
orange    4
mango     4
grape     3
Name: count, dtype: int64


In [69]:
print(df.describe(include="all"))

                                 date  stall  fruit        qty      price
count                              23     23     23  23.000000  23.000000
unique                            NaN      2      5        NaN        NaN
top                               NaN  North  apple        NaN        NaN
freq                              NaN     12      7        NaN        NaN
mean    2024-06-03 06:15:39.130434816    NaN    NaN  10.260870  27.391304
min               2024-03-06 00:00:00    NaN    NaN   3.000000  10.000000
25%               2024-06-05 00:00:00    NaN    NaN   7.000000  20.000000
50%               2024-06-07 00:00:00    NaN    NaN   9.000000  30.000000
75%               2024-06-09 12:00:00    NaN    NaN  12.500000  35.000000
max               2024-06-12 00:00:00    NaN    NaN  25.000000  50.000000
std                               NaN    NaN    NaN   5.327607  12.510865


In [ ]:
| Issue | Where (column) | How many | What I did and why |
|---------|---------|---------|---------|
| Price stored as text (₹ symbol) | price | 25 rows | Removed ₹ symbol and converted to numeric |
| Quantity stored as text | qty | 25 rows | Converted to numeric using to_numeric() |
| Date stored as text | date | 25 rows | Converted to datetime format |
| Duplicate row | Entire row | 1 row | Removed using drop_duplicates() |
| Inconsistent stall names | stall | 6 spellings | Standardised with strip() and title() |
| Inconsistent fruit names | fruit | Multiple entries | Standardised with strip() and lower() |
| Typo "appel" | fruit | 1 row | Replaced with "apple" |
| Missing date | date | 1 row | Dropped row because date is required |
| Missing quantity | qty | 2 rows | Filled with median quantity |
| Missing price | price | 1 row | Filled with median price |
| Negative quantity (-5) | qty | 1 row | Treated as an error and replaced |
| Extreme quantity (9999) | qty | 1 row | Treated as a data-entry error and replaced |
| Quantity 25 flagged by IQR | qty | 1 row | Kept because it is a valid large sale |
| Validation failure | qty | 4 rows | Fixed invalid and missing quantities |